# 05｜儀表板與報告自動化（對應課程 M8、M9）

**本 Notebook 的目標**

1. 計算「最近一個**完整**月份」的 KPI，並和前一個月比較
2. 用 matplotlib 做一張「營運健康儀表板」：KPI 列＋四張圖，每張圖一句重點
3. 用程式自動產出**一頁式 HTML 報告**：標題就是結論、三個重點、建議行動、定義與資料期間
4. （選用）用 python-pptx 產出 5 頁簡報

> **AI 做法 vs 傳統做法**：Claude Dashboards 一句話就能做出可刷新的儀表板；
> 傳統做法要自己寫每張圖，但**每個數字、每句結論都由程式從資料算出來**，下個月重跑就自動更新。

In [ ]:
import base64
from io import BytesIO
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import islemart_lib as L

L.setup_chinese_font()
BLUE, ORANGE, GRAY, INK = "#2E7D9A", "#E07A3F", "#9AA5AD", "#172A3A"
pd.set_option("display.float_format", "{:,.4f}".format)

T = L.load_all()
wide = L.build_orders_wide(T)

## 1. 哪個月份才算「完整」？

資料最後幾天還有很多訂單在途中。規則：**該月已結案（送達、取消、無法出貨）的訂單 ≥ 95%** 才算完整月份。
這樣 KPI 就不會因為資料還沒進來而「看起來下滑」。

In [ ]:
closed = wide["order_status"].isin(["delivered", "canceled", "unavailable"])
closed_share = closed.groupby(wide["purchase_month"]).mean()
complete_months = closed_share[closed_share >= 0.95].index
CUR = complete_months.max()
PREV = complete_months[complete_months < CUR].max()
print(f"最近一個完整月份：{CUR}（前一個月：{PREV}）")
closed_share.tail(3).map("{:.1%}".format)

## 2. KPI：本月 vs 上月

四個指標都呼叫 `islemart_lib` 裡的指標定義函式，確保和其他 Notebook 口徑一致。

In [ ]:
def kpis(df: pd.DataFrame) -> dict:
    return {"訂單數": len(df), "營收": L.revenue(df),
            "延遲率": L.late_rate(df), "平均評分": L.avg_score(df)}

cur, prev = kpis(wide[wide["purchase_month"] == CUR]), kpis(wide[wide["purchase_month"] == PREV])
kpi = pd.DataFrame({CUR: cur, PREV: prev})
kpi["變化"] = [f"{(cur[k] / prev[k] - 1):+.1%}" if k in ("訂單數", "營收")
               else f"{(cur[k] - prev[k]) * 100:+.1f} 個百分點" if k == "延遲率"
               else f"{cur[k] - prev[k]:+.2f} 分" for k in kpi.index]
kpi

## 3. 四張圖的資料，以及每張圖的「一句重點」

重點句由程式依資料產生，不是手寫：資料變了，句子跟著變。

In [ ]:
# 圖 1：每月訂單數
monthly = wide.groupby("purchase_month").size()
peak_m = monthly.idxmax()
note1 = f"{peak_m} 訂單最多（{monthly.max():,} 筆），是其他月份平均的 {monthly.max() / monthly.drop(peak_m).mean():.1f} 倍"

# 圖 2：各縣市延遲率（本月）
cm = wide[(wide["purchase_month"] == CUR) & wide["is_valid_delivery"]]
county_late = cm.groupby("customer_county")["is_late"].mean().mul(100).sort_values()
note2 = f"{county_late.index[-1]}延遲率最高（{county_late.iloc[-1]:.1f}%），最低的{county_late.index[0]}只有 {county_late.iloc[0]:.1f}%"

# 圖 3：各類別平均評分（本月）
cat_score = wide[wide["purchase_month"] == CUR].groupby("main_category")["review_score"].mean().sort_values()
note3 = f"{cat_score.index[0]}評分最低（{cat_score.iloc[0]:.2f}），{cat_score.index[-1]}最高（{cat_score.iloc[-1]:.2f}）"

# 圖 4：每月低分（1–2 分）評論占比
low = wide.dropna(subset=["review_score"]).groupby("purchase_month")["review_score"].apply(lambda s: (s <= 2).mean() * 100)
low = low.loc[:CUR]
note4 = f"{CUR} 低分占比 {low[CUR]:.1f}%，兩年平均 {low.mean():.1f}%"

for n in (note1, note2, note3, note4):
    print("・", n)

## 4. 畫成儀表板

版面：最上面一列 KPI，下面 2×2 四張圖；每張圖的標題下方放重點句。

In [ ]:
def fmt_value(k, v):
    return {"訂單數": f"{v:,.0f}", "營收": f"{v / 1e6:,.2f} 百萬", "延遲率": f"{v:.1%}", "平均評分": f"{v:.2f}"}[k]

fig = plt.figure(figsize=(16, 11), facecolor="#FBFAF7")
gs = fig.add_gridspec(3, 4, height_ratios=[0.55, 1, 1], hspace=0.55, wspace=0.35)
fig.suptitle(f"IsleMart 營運健康儀表板　｜　{CUR}（與 {PREV} 比較）", x=0.06, y=0.95, ha="left",
             fontsize=18, fontweight="bold", color=INK)

for i, k in enumerate(kpi.index):
    ax = fig.add_subplot(gs[0, i]); ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, color="white", ec="#E4DED2"))
    ax.text(0.06, 0.78, k, fontsize=12, color="#6B7680", transform=ax.transAxes)
    ax.text(0.06, 0.38, fmt_value(k, cur[k]), fontsize=22, fontweight="bold", color=INK, transform=ax.transAxes)
    ax.text(0.06, 0.10, f"vs 上月 {kpi.loc[k, '變化']}", fontsize=11, color="#6B7680", transform=ax.transAxes)

def style(ax, title, note):
    ax.set_title(f"{title}\n{note}", loc="left", fontsize=12, color=INK, linespacing=1.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=9)

ax1 = fig.add_subplot(gs[1, :2])
ax1.bar(monthly.index, monthly.values, color=[ORANGE if m == peak_m else BLUE for m in monthly.index], width=0.7)
ax1.set_xticks(monthly.index[::3]); style(ax1, "每月訂單數", note1)

ax2 = fig.add_subplot(gs[1, 2:])
top = county_late.tail(10)
ax2.barh(top.index, top.values, color=[ORANGE if i == len(top) - 1 else BLUE for i in range(len(top))], height=0.6)
ax2.set_xlabel("延遲率（%）"); style(ax2, f"延遲率最高的 10 個縣市（{CUR}）", note2)

ax3 = fig.add_subplot(gs[2, :2])
ax3.barh(cat_score.index, cat_score.values, color=[ORANGE if i == 0 else BLUE for i in range(len(cat_score))], height=0.6)
ax3.set_xlim(cat_score.min() - 0.5, cat_score.max() + 0.3); ax3.set_xlabel("平均評分（1–5）")
style(ax3, f"各類別平均評分（{CUR}）", note3)

ax4 = fig.add_subplot(gs[2, 2:])
ax4.plot(low.index, low.values, color=BLUE, linewidth=2, marker="o", markersize=3)
ax4.set_ylim(0, max(low) * 1.3); ax4.set_xticks(low.index[::3]); ax4.set_ylabel("%")
style(ax4, "每月低分（1–2 分）評論占比", note4)

fig.text(0.06, 0.015, "定義：營收＝已送達訂單商品售價（不含運費、排除價格異常）；延遲率＝有效送達訂單中晚於預計到貨日的比例。"
         "資料：IsleMart 模擬資料。", fontsize=9, color="#6B7680")
fig.savefig("dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

**注意第 3 張圖**：評分軸不是從 0 開始，是為了看出類別之間的差距。這種情況一定要在軸上標清楚範圍，查核清單的「座標軸」那一項就是在檢查這件事。

## 5. 一頁式報告：結論先行

依 M9 的金字塔原則，用程式組出報告：

| 位置 | 內容 | 從哪裡來 |
| --- | --- | --- |
| 標題 | 一句結論，含數字 | KPI 與縣市延遲率 |
| 三個重點 | 每點一個數字＋一句「所以呢」 | 第 3 節的重點句 |
| 主圖 | 儀表板圖 | `dashboard.png` |
| 建議行動 | 兩項，含負責單位 | 依資料套用模板 |
| 註腳 | 指標定義與資料期間 | 固定文字 |

In [ ]:
worst_county, worst_rate = county_late.index[-1], county_late.iloc[-1]
title = (f"{CUR} 訂單 {cur['訂單數']:,} 筆（{kpi.loc['訂單數', '變化']}），"
         f"但延遲率升至 {cur['延遲率']:.1%}，{worst_county}最嚴重")

points = [
    (f"訂單：{note1}", "檔期前兩週就要備妥物流與客服人力。"),
    (f"配送：{note2}", "延遲集中在特定地區，優先檢討這些地區的承諾到貨天數。"),
    (f"評分：{note3}", "低分類別要和延遲一起看，先排除物流因素再檢討商品。"),
]
actions = [
    ("物流部", f"兩週內檢討{worst_county}等延遲率前三名縣市的承諾到貨天數"),
    ("營運部", "下一檔期前完成延遲率最高 10 家賣家的出貨輔導"),
]
print(title)

In [ ]:
img64 = base64.b64encode(Path("dashboard.png").read_bytes()).decode()
points_html = "".join(f"<li><b>{p}</b><br><span class='so'>所以呢：{s}</span></li>" for p, s in points)
actions_html = "".join(f"<li><b>{who}</b>｜{what}</li>" for who, what in actions)

html = f'''<!doctype html>
<html lang="zh-Hant"><head><meta charset="utf-8"><title>IsleMart {CUR} 營運報告</title>
<style>
 body {{ font-family: "Noto Sans TC","Microsoft JhengHei","PingFang TC",sans-serif; max-width: 980px;
        margin: 32px auto; padding: 0 16px; color: #3D4A55; line-height: 1.6; }}
 h1 {{ color: #172A3A; font-size: 26px; }}
 .so {{ color: #B4561C; }}
 img {{ width: 100%; border: 1px solid #E4DED2; }}
 footer {{ font-size: 13px; color: #6B7680; border-top: 1px solid #E4DED2; margin-top: 24px; padding-top: 8px; }}
</style></head><body>
<p style="color:#6B7680">IsleMart 營運週期報告｜{CUR}</p>
<h1>{title}</h1>
<h2>三個重點</h2><ol>{points_html}</ol>
<img src="data:image/png;base64,{img64}" alt="IsleMart 營運健康儀表板">
<h2>建議行動</h2><ul>{actions_html}</ul>
<footer>定義：營收＝已送達訂單商品售價，不含運費、不扣折價券、排除價格異常；延遲率＝有效送達訂單中送達日晚於預計到貨日的比例；
平均評分＝每筆訂單一則評論（已去重）。資料期間：{CUR}（比較 {PREV}）。資料來源：IsleMart 模擬資料。</footer>
</body></html>'''

Path("report.html").write_text(html, encoding="utf-8")
print("已產生 report.html，用瀏覽器打開即可；要 PDF 可用瀏覽器「列印 → 另存為 PDF」。")

## 6.（選用）用 python-pptx 產出 5 頁簡報

先安裝：`pip install python-pptx`。沒安裝的話，這一格會略過。

In [ ]:
try:
    from pptx import Presentation
    from pptx.util import Inches, Pt

    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
    blank = prs.slide_layouts[6]

    def add_text(slide, text, top, size, bold=False):
        box = slide.shapes.add_textbox(Inches(0.7), Inches(top), Inches(12), Inches(1))
        box.text_frame.word_wrap = True
        p = box.text_frame.paragraphs[0]
        p.text, p.font.size, p.font.bold = text, Pt(size), bold
        return box

    s = prs.slides.add_slide(blank); add_text(s, f"IsleMart {CUR} 營運報告", 2.5, 40, True); add_text(s, title, 3.6, 22)
    s = prs.slides.add_slide(blank); add_text(s, "KPI：本月 vs 上月", 0.5, 30, True)
    for i, k in enumerate(kpi.index):
        add_text(s, f"{k}　{fmt_value(k, cur[k])}　（{kpi.loc[k, '變化']}）", 1.8 + i * 1.0, 24)
    s = prs.slides.add_slide(blank); add_text(s, "營運健康儀表板", 0.4, 28, True)
    s.shapes.add_picture("dashboard.png", Inches(1.2), Inches(1.3), height=Inches(5.9))
    s = prs.slides.add_slide(blank); add_text(s, "三個重點", 0.5, 30, True)
    for i, (p, so) in enumerate(points):
        add_text(s, f"{i + 1}. {p}\n　所以呢：{so}", 1.6 + i * 1.7, 20)
    s = prs.slides.add_slide(blank); add_text(s, "建議行動", 0.5, 30, True)
    for i, (who, what) in enumerate(actions):
        add_text(s, f"{who}｜{what}", 1.8 + i * 1.2, 22)
    prs.save("report.pptx")
    print("已產生 report.pptx（5 頁）")
except ImportError:
    print("沒有安裝 python-pptx，略過簡報產生。")

## 小結

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| Claude Dashboards 一句話建儀表板 | matplotlib GridSpec 排版，KPI＋四張圖 |
| 每張圖附 SQL 與刷新時間 | 每個數字來自同一套指標函式，重跑即更新 |
| 請 AI 把標題改成結論 | 結論句由程式依資料組出 |
| Claude Docs／Slides、ChatGPT Work 產出報告 | HTML 模板＋python-pptx |

**差異**：傳統做法初次建置較花時間，但完全可控、可重現；AI 做法快，適合探索。
兩者並用：用 AI 找到值得追的問題，再把確定要長期追蹤的指標寫成程式（Notebook 06）。